In [1]:
# Install dependencies if needed
# !pip install requests pandas python-dotenv

import os
import time
import requests
import pandas as pd
from datetime import date, timedelta

In [2]:
#API_KEY = os.getenv("MASSIVE_API_KEY", "")

# Optional temporary method. Uncomment and paste your key if you have not set an environment variable.
API_KEY = "xsJlMe_F2PsGLr7EJsOHOBP0VpMb7Rsy"

if not API_KEY:
    raise ValueError("No API key found. Set MASSIVE_API_KEY or paste it into API_KEY above.")

BASE_URL = "https://api.massive.com"
print("API key loaded. Length:", len(API_KEY))

API key loaded. Length: 32


In [3]:
def massive_get(path, params=None, sleep_seconds=0.0):
    # Simple Massive API GET helper.
    if params is None:
        params = {}
    params = dict(params)
    params["apiKey"] = API_KEY
    url = BASE_URL + path
    response = requests.get(url, params=params, timeout=30)
    print("GET", response.url.replace(API_KEY, "***API_KEY***"))
    print("Status:", response.status_code)
    
    if sleep_seconds:
        time.sleep(sleep_seconds)
    
    try:
        data = response.json()
    except Exception:
        print(response.text[:500])
        response.raise_for_status()
        return None
    
    if response.status_code >= 400:
        print(data)
        response.raise_for_status()
    
    return data


def results_to_df(data):
    # Convert common Massive response format into a DataFrame.
    if not data:
        return pd.DataFrame()
    results = data.get("results", [])
    if isinstance(results, dict):
        results = [results]
    return pd.DataFrame(results)

In [4]:
index_tickers_to_test = ["I:SPX", "I:VIX"]

for ticker in index_tickers_to_test:
    print("\n--- Checking", ticker, "---")
    data = massive_get("/v3/reference/tickers", params={"ticker": ticker, "market": "indices"}, sleep_seconds=1)
    df = results_to_df(data)
    display(df.head())


--- Checking I:SPX ---
GET https://api.massive.com/v3/reference/tickers?ticker=I%3ASPX&market=indices&apiKey=***API_KEY***
Status: 200


,ticker,name,market,locale,active,source_feed
0,I:SPX,Standard & Poor's 500,indices,us,True,CboeGlobalIndicesMain



--- Checking I:VIX ---
GET https://api.massive.com/v3/reference/tickers?ticker=I%3AVIX&market=indices&apiKey=***API_KEY***
Status: 200


,ticker,name,market,locale,active,source_feed
0,I:VIX,Cboe Volatility Index,indices,us,True,CboeGlobalIndicesMain


In [5]:
# Change these dates to a recent weekday if the request returns no data.
from_date = "2026-06-01"
to_date = "2026-06-01"

for ticker in ["I:SPX", "I:VIX"]:
    print("\n--- Minute aggregates for", ticker, "---")
    path = f"/v2/aggs/ticker/{ticker}/range/1/minute/{from_date}/{to_date}"
    data = massive_get(path, params={"adjusted": "true", "sort": "asc", "limit": 5000}, sleep_seconds=1)
    df = results_to_df(data)
    print("Rows:", len(df))
    display(df.head())


--- Minute aggregates for I:SPX ---
GET https://api.massive.com/v2/aggs/ticker/I:SPX/range/1/minute/2026-06-01/2026-06-01?adjusted=true&sort=asc&limit=5000&apiKey=***API_KEY***
Status: 403
{'status': 'NOT_AUTHORIZED', 'request_id': 'd4769e0d2f8044baad5c159dd53f43e6', 'message': 'You are not entitled to this data. Please upgrade your plan at https://massive.com/pricing'}


HTTPError: 403 Client Error: Forbidden for url: https://api.massive.com/v2/aggs/ticker/I:SPX/range/1/minute/2026-06-01/2026-06-01?adjusted=true&sort=asc&limit=5000&apiKey=xsJlMe_F2PsGLr7EJsOHOBP0VpMb7Rsy